<a href="https://colab.research.google.com/github/Arsen-Almasov/OC11/blob/main/task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile faults.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <sys/resource.h>

#define ROWS 8192
#define COLS 8192

void print_faults(const char *label) {
    struct rusage usage;
    getrusage(RUSAGE_SELF, &usage);

    // Чтение /proc/self/stat (поля 10 - minflt, 12 - majflt)
    FILE *f = fopen("/proc/self/stat", "r");
    unsigned long minflt = 0, majflt = 0;
    if (f) {
        fscanf(f, "%*d %*s %*c %*d %*d %*d %*d %*d %*u %lu %*u %lu", &minflt, &majflt);
        fclose(f);
    }

    printf("[%s]\n", label);
    printf("  getrusage -> minor: %ld, major: %ld\n", usage.ru_minflt, usage.ru_majflt);
    printf("  /proc     -> minor: %lu, major: %lu\n\n", minflt, majflt);
}

int main() {
    // Выделение большого массива (~256 МБ)
    int **arr = malloc(ROWS * sizeof(int *));
    for (int i = 0; i < ROWS; i++) {
        arr[i] = malloc(COLS * sizeof(int));
    }

    print_faults("После malloc (до обращения к памяти)");

    // 1. Обход по строкам (by rows)
    struct timespec start, end;
    clock_gettime(CLOCK_MONOTONIC, &start);
    for (int i = 0; i < ROWS; i++) {
        for (int j = 0; j < COLS; j++) {
            arr[i][j] = i + j;
        }
    }
    clock_gettime(CLOCK_MONOTONIC, &end);
    double time_rows = (end.tv_sec - start.tv_sec) + (end.tv_nsec - start.tv_nsec) / 1e9;
    printf("by rows:    %.3f s\n", time_rows);
    print_faults("После обхода по строкам");

    // 2. Обход по столбцам (by columns)
    clock_gettime(CLOCK_MONOTONIC, &start);
    for (int j = 0; j < COLS; j++) {
        for (int i = 0; i < ROWS; i++) {
            arr[i][j] = i - j;
        }
    }
    clock_gettime(CLOCK_MONOTONIC, &end);
    double time_cols = (end.tv_sec - start.tv_sec) + (end.tv_nsec - start.tv_nsec) / 1e9;
    printf("by columns: %.3f s (x%.1f)\n", time_cols, time_cols / time_rows);
    print_faults("После обхода по столбцам");

    for (int i = 0; i < ROWS; i++) free(arr[i]);
    free(arr);
    return 0;
}


Writing faults.c


In [2]:
!gcc -O0 faults.c -o faults
!./faults

[После malloc (до обращения к памяти)]
  getrusage -> minor: 8389, major: 0
  /proc     -> minor: 8397, major: 0

by rows:    0.341 s
[После обхода по строкам]
  getrusage -> minor: 65776, major: 0
  /proc     -> minor: 65776, major: 0

by columns: 0.881 s (x2.6)
[После обхода по столбцам]
  getrusage -> minor: 65776, major: 0
  /proc     -> minor: 65776, major: 0



In [4]:
! python3 replace_solution.py
= Контрольная строка, 3 кадра =
FIFO: 15 отказов
LRU: 12 отказов
Optimal: 9 отказов
Clock: 14 отказов


SyntaxError: invalid syntax (925346962.py, line 2)

! python3 replace_solution.py
= Контрольная строка, 3 кадра =
FIFO: 15 отказов
LRU: 12 отказов
Optimal: 9 отказов
Clock: 14 отказов


In [5]:
def fifo_trace(pages, capacity):
    frames = []  # Текущее состояние кадров памяти
    queue = []   # Очередь для отслеживания порядка загрузки (FIFO)
    page_faults = 0

    for step, page in enumerate(pages, start=1):
        status = ""
        if page not in frames:
            page_faults += 1
            status = "PF"
            if len(frames) < capacity:
                frames.append(page)
                queue.append(page)
            else:
                # Выбираем страницу, которая пришла первой
                oldest = queue.pop(0)
                idx = frames.index(oldest)
                # Заменяем её на том же месте в списке кадров
                frames[idx] = page
                queue.append(page)

        # Форматируем вывод кадров шириной в 5 символов, как в терминале на фото
        frames_str = " ".join(map(str, frames)).ljust(5)
        print(f"{step:2d}  обр={page}  [{frames_str}]  {status}")

    print(f"\nВсего страничных отказов (PF): {page_faults}")

if __name__ == "__main__":
    # Последовательность обращений со слайда (шаги 1-11)
    reference_string = [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0]
    frames_count = 3

    fifo_trace(reference_string, frames_count)


 1  обр=7  [7    ]  PF
 2  обр=0  [7 0  ]  PF
 3  обр=1  [7 0 1]  PF
 4  обр=2  [2 0 1]  PF
 5  обр=0  [2 0 1]  
 6  обр=3  [2 3 1]  PF
 7  обр=0  [2 3 0]  PF
 8  обр=4  [4 3 0]  PF
 9  обр=2  [4 2 0]  PF
10  обр=3  [4 2 3]  PF
11  обр=0  [0 2 3]  PF

Всего страничных отказов (PF): 10
